# M-MMLU ↔ RankMe Correlation

Evaluates multilingual LLMs on **Multilingual MMLU (M-MMLU)** across languages and
correlates per-language accuracy with the **RankMe** score the model obtains in that
language (computed from its internal representations by the main pipeline).

**Configuration** lives entirely in `configs/giacomo.yaml`:
- Which models to evaluate and where their RankMe CSVs are
- Which languages and lm-eval task names to use
- Which checkpoint / layer / aggregation to extract RankMe from
- Evaluation hyperparameters (few-shot count, batch size, limit, device)

M-MMLU results are **cached** in `results/mmmlu/` so re-running the notebook is cheap.
Set `FORCE_RERUN = True` in the evaluation cell to redo inference.

In [ ]:
import gc
import os
import sys
import json
import yaml
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

# Resolve project root whether the notebook is run from notebooks/ or from root
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"Notebook dir : {NOTEBOOK_DIR}")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Override by setting env var GIACOMO_CONFIG=/path/to/other.yaml
CONFIG_PATH = Path(os.environ.get(
    "GIACOMO_CONFIG",
    NOTEBOOK_DIR / "configs" / "giacomo.yaml"
))

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

models_cfg = config["models"]
eval_cfg   = config["evaluation"]
rankme_cfg = config["rankme"]
output_cfg = config.get("output", {})
languages  = eval_cfg["languages"]

results_dir = PROJECT_ROOT / output_cfg.get("results_dir", "results/mmmlu")
figures_dir = PROJECT_ROOT / output_cfg.get("figures_dir", "results/figures")
results_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

print(f"Config       : {CONFIG_PATH}")
print(f"Models       : {[m['name'] for m in models_cfg]}")
print(f"Languages    : {[l['name'] for l in languages]}")
print(f"RankMe config: checkpoint={rankme_cfg['checkpoint']!r}, "
      f"layer={rankme_cfg['layer']!r}, agg={rankme_cfg['aggregation']!r}")
print(f"Results dir  : {results_dir}")
print(f"Figures dir  : {figures_dir}")

## 1. Load RankMe Scores

Each model's RankMe CSV (produced by `geometry_analysis/geometry_analysis.py`) is filtered
to the configured **checkpoint / layer / aggregation** to obtain one score per language.

In [ ]:
def _resolve_checkpoint(df: pd.DataFrame, checkpoint: str, model_name: str) -> str:
    """Return the checkpoint string to filter on, handling 'auto' mode."""
    if checkpoint != "auto":
        return checkpoint
    if "main" in df["checkpoint"].values:
        resolved = "main"
    else:
        resolved = df["checkpoint"].iloc[-1]
    print(f"[{model_name}] auto checkpoint → '{resolved}'")
    return resolved


def load_rankme_scores(
    model_cfg: dict,
    rankme_cfg: dict,
    project_root: Path,
) -> pd.DataFrame:
    """
    Load RankMe scores for one model.
    Returns a DataFrame with columns [language, rankme] for the configured
    checkpoint/layer/aggregation, or an empty DataFrame on failure.
    """
    path = project_root / model_cfg["rankme_path"]
    name = model_cfg["name"]

    if not path.exists():
        print(f"[{name}] RankMe file not found: {path}")
        return pd.DataFrame(columns=["language", "rankme"])

    df = pd.read_csv(path)
    ckpt = _resolve_checkpoint(df, rankme_cfg["checkpoint"], name)
    layer = rankme_cfg["layer"]
    agg   = rankme_cfg["aggregation"]

    mask = (
        (df["checkpoint"] == ckpt) &
        (df["layer"]      == layer) &
        (df["aggregation"] == agg)
    )
    filtered = df[mask].copy()

    if filtered.empty:
        print(f"[{name}] No rows for checkpoint='{ckpt}', layer='{layer}', agg='{agg}'")
        print(f"  Available checkpoints : {sorted(df['checkpoint'].unique())}")
        print(f"  Available layers      : {sorted(df['layer'].unique())}")
        print(f"  Available aggregations: {sorted(df['aggregation'].unique())}")
        return pd.DataFrame(columns=["language", "rankme"])

    return filtered[["dataset", "rankme"]].rename(columns={"dataset": "language"})


def load_all_rankme(
    model_cfg: dict,
    checkpoint: str,
    project_root: Path,
) -> pd.DataFrame:
    """
    Load all layers × aggregations for one model at one checkpoint.
    Used for the layer-wise correlation analysis.
    """
    path = project_root / model_cfg["rankme_path"]
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path)
    ckpt = _resolve_checkpoint(df, checkpoint, model_cfg["name"])
    return df[df["checkpoint"] == ckpt].copy()


# ── Load primary RankMe scores ─────────────────────────────────────────────────
rankme_dfs: dict[str, pd.DataFrame] = {}
for mcfg in models_cfg:
    df = load_rankme_scores(mcfg, rankme_cfg, PROJECT_ROOT)
    rankme_dfs[mcfg["name"]] = df
    if not df.empty:
        print(f"\n{mcfg['name']} — RankMe (ckpt={rankme_cfg['checkpoint']}, "
              f"layer={rankme_cfg['layer']}, agg={rankme_cfg['aggregation']}):")
        for _, row in df.iterrows():
            print(f"  {row['language']:<15}: {row['rankme']:.3f}")
    else:
        print(f"\n{mcfg['name']}: no RankMe scores loaded.")

## 2. M-MMLU Evaluation

Uses [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) to evaluate
each model on all configured languages in a single pass (model loaded once).

Results are **cached** as JSON in `results/mmmlu/`. Set `FORCE_RERUN = True` to redo.

Install the harness if needed:
```
pip install lm-eval
```

In [ ]:
try:
    from lm_eval import simple_evaluate
    HAS_LM_EVAL = True
    print("lm-evaluation-harness is available.")
except ImportError:
    HAS_LM_EVAL = False
    print("lm-evaluation-harness not installed — inference will be skipped.")
    print("Install with: pip install lm-eval")
    print("Cached results will still be loaded if they exist.")

In [ ]:
def _extract_acc(task_res: dict) -> tuple[float | None, float | None]:
    """Extract accuracy and stderr from a task result dict (handles lm-eval key variants)."""
    acc = (
        task_res.get("acc,none") or
        task_res.get("acc") or
        task_res.get("acc_norm,none") or
        task_res.get("acc_norm")
    )
    stderr = (
        task_res.get("acc_stderr,none") or
        task_res.get("acc_stderr") or
        task_res.get("acc_norm_stderr,none") or
        task_res.get("acc_norm_stderr")
    )
    return (float(acc) if acc is not None else None,
            float(stderr) if stderr is not None else None)


def run_mmmlu(
    model_cfg: dict,
    eval_cfg: dict,
    results_dir: Path,
    force_rerun: bool = False,
) -> dict[str, dict]:
    """
    Evaluate one model on all configured M-MMLU languages.
    Loads the model once and evaluates all tasks in a single pass.
    Results are cached; pass force_rerun=True to re-run.

    Returns a dict: {language_name: {accuracy, stderr, task}} 
    """
    model_name = model_cfg["name"]
    safe_name  = model_name.lower().replace(" ", "_").replace("/", "_").replace("-", "_")
    cache_path = results_dir / f"mmmlu_{safe_name}.json"

    if cache_path.exists() and not force_rerun:
        print(f"[{model_name}] Loading cached results: {cache_path}")
        with open(cache_path) as f:
            return json.load(f)

    if not HAS_LM_EVAL:
        print(f"[{model_name}] Skipping (lm-eval not installed).")
        return {}

    tasks         = [l["lm_eval_task"] for l in eval_cfg["languages"]]
    task_to_lang  = {l["lm_eval_task"]: l["name"] for l in eval_cfg["languages"]}

    model_args = (
        f"pretrained={model_cfg['hf_id']},"
        f"revision={model_cfg.get('revision', 'main')},"
        f"dtype={model_cfg.get('torch_dtype', 'bfloat16')},"
        f"trust_remote_code={str(model_cfg.get('trust_remote_code', True)).lower()}"
    )

    print(f"[{model_name}] Running M-MMLU on {len(tasks)} tasks (model loaded once)...")

    out = simple_evaluate(
        model="hf",
        model_args=model_args,
        tasks=tasks,
        num_fewshot=eval_cfg.get("num_fewshot", 5),
        batch_size=eval_cfg.get("batch_size", 4),
        device=eval_cfg.get("device", "cuda"),
        limit=eval_cfg.get("limit"),
        log_samples=False,
    )

    results: dict[str, dict] = {}
    for task, lang_name in task_to_lang.items():
        task_res = out["results"].get(task, {})
        acc, stderr = _extract_acc(task_res)
        results[lang_name] = {"accuracy": acc, "stderr": stderr, "task": task}
        status = f"{acc:.4f}" if acc is not None else "N/A"
        print(f"  {lang_name:<15}: {status}")

    with open(cache_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"[{model_name}] Results cached → {cache_path}")

    gc.collect()
    return results

In [ ]:
FORCE_RERUN = False  # ← set True to re-run inference even when a cache exists

mmlu_results: dict[str, dict] = {}
for mcfg in models_cfg:
    print(f"\n{'='*55}\nModel: {mcfg['name']}\n{'='*55}")
    mmlu_results[mcfg["name"]] = run_mmmlu(mcfg, eval_cfg, results_dir, FORCE_RERUN)

# Summary table
lang_names = [l["name"] for l in languages]
summary = {"Language": lang_names}
for mcfg in models_cfg:
    mname = mcfg["name"]
    summary[mname] = [
        (mmlu_results.get(mname, {}).get(l) or {}).get("accuracy")
        for l in lang_names
    ]
df_mmlu = pd.DataFrame(summary).set_index("Language")
print("\nM-MMLU Accuracy Summary:")
print(df_mmlu.to_string(float_format="{:.4f}".format))

## 3. Correlation Analysis

For each model, we compute the **Pearson** and **Spearman** correlation between:
- RankMe score in language L (from the model's internal representations)
- M-MMLU accuracy in language L (downstream evaluation)

A positive correlation would mean higher-quality representations (high RankMe) → better downstream performance.

In [ ]:
# ── Build merged DataFrame ─────────────────────────────────────────────────────
rows = []
for mcfg in models_cfg:
    mname  = mcfg["name"]
    rm_df  = rankme_dfs.get(mname, pd.DataFrame(columns=["language", "rankme"]))
    rm_map = dict(zip(rm_df["language"], rm_df["rankme"])) if not rm_df.empty else {}

    for lang in languages:
        lname   = lang["name"]
        rm_val  = rm_map.get(lname)
        mmlu_r  = (mmlu_results.get(mname, {}).get(lname) or {})
        acc_val = mmlu_r.get("accuracy")
        rows.append({
            "model":         mname,
            "language":      lname,
            "rankme":        rm_val,
            "mmlu_accuracy": acc_val,
            "has_both":      rm_val is not None and acc_val is not None,
        })

df_merged = pd.DataFrame(rows)

# Completeness overview
pivot = df_merged.pivot_table(
    values="has_both", index="language", columns="model", aggfunc="first"
)
print("Data completeness (True = both RankMe and MMLU available):")
print(pivot.to_string())

df_valid = df_merged[df_merged["has_both"]].copy()
print(f"\nValid pairs for correlation: {len(df_valid)} / {len(df_merged)}")

In [ ]:
# ── Compute correlations ───────────────────────────────────────────────────────
corr_results: dict[str, dict] = {}

for mname in df_valid["model"].unique():
    sub = df_valid[df_valid["model"] == mname]
    n   = len(sub)

    if n < 3:
        print(f"{mname}: only {n} data point(s) — skipping (need ≥ 3).")
        continue

    pr, pp = stats.pearsonr(sub["rankme"], sub["mmlu_accuracy"])
    sr, sp = stats.spearmanr(sub["rankme"], sub["mmlu_accuracy"])

    corr_results[mname] = {
        "pearson_r":  pr, "pearson_p":  pp,
        "spearman_r": sr, "spearman_p": sp,
        "n": n,
    }
    print(f"{mname}  (n={n}):")
    print(f"  Pearson   r = {pr:+.4f}   p = {pp:.4f}{'  *' if pp < 0.05 else ''}")
    print(f"  Spearman  r = {sr:+.4f}   p = {sp:.4f}{'  *' if sp < 0.05 else ''}")

# Pooled across all models
if len(df_valid) >= 3:
    pr, pp = stats.pearsonr(df_valid["rankme"], df_valid["mmlu_accuracy"])
    sr, sp = stats.spearmanr(df_valid["rankme"], df_valid["mmlu_accuracy"])
    corr_results["Pooled"] = {
        "pearson_r":  pr, "pearson_p":  pp,
        "spearman_r": sr, "spearman_p": sp,
        "n": len(df_valid),
    }
    print(f"\nPooled  (n={len(df_valid)}):")
    print(f"  Pearson   r = {pr:+.4f}   p = {pp:.4f}")
    print(f"  Spearman  r = {sr:+.4f}   p = {sp:.4f}")

## 4. Visualization

In [ ]:
# ── Scatter: RankMe vs M-MMLU accuracy ────────────────────────────────────────
if df_valid.empty:
    print("No valid data for plotting.")
else:
    model_list  = list(df_valid["model"].unique())
    n_models    = len(model_list)
    color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5), squeeze=False)

    for i, mname in enumerate(model_list):
        ax  = axes[0, i]
        sub = df_valid[df_valid["model"] == mname]
        cr  = corr_results.get(mname, {})

        ax.scatter(sub["rankme"], sub["mmlu_accuracy"],
                   s=80, alpha=0.85, color=color_cycle[i])

        for _, row in sub.iterrows():
            ax.annotate(row["language"],
                        (row["rankme"], row["mmlu_accuracy"]),
                        xytext=(4, 3), textcoords="offset points", fontsize=8)

        if len(sub) >= 2:
            m, b = np.polyfit(sub["rankme"], sub["mmlu_accuracy"], 1)
            x_lo, x_hi = sub["rankme"].min(), sub["rankme"].max()
            xs = np.linspace(x_lo, x_hi, 50)
            ax.plot(xs, m * xs + b, "--", color=color_cycle[i], alpha=0.5, linewidth=1.5)

        pr  = cr.get("pearson_r",  float("nan"))
        sr  = cr.get("spearman_r", float("nan"))
        n   = cr.get("n", 0)
        ax.set_title(f"{mname}  (n={n})\nPearson r={pr:+.3f}   Spearman r={sr:+.3f}",
                     fontsize=10)
        ax.set_xlabel(
            f"RankMe  ({rankme_cfg['layer']}, {rankme_cfg['aggregation']})",
            fontsize=9
        )
        ax.set_ylabel("M-MMLU Accuracy", fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.suptitle(
        f"RankMe vs M-MMLU — checkpoint: {rankme_cfg['checkpoint']}",
        fontsize=12, y=1.02
    )
    plt.tight_layout()
    fig_path = figures_dir / (
        f"rankme_vs_mmmlu_scatter"
        f"_{rankme_cfg['layer']}_{rankme_cfg['aggregation']}.pdf"
    )
    plt.savefig(fig_path, bbox_inches="tight")
    plt.show()
    print(f"Saved → {fig_path}")

In [ ]:
# ── Rank comparison: Spearman rank scatter ─────────────────────────────────────
if not df_valid.empty:
    model_list = list(df_valid["model"].unique())
    n_models   = len(model_list)

    fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 5), squeeze=False)

    for i, mname in enumerate(model_list):
        ax  = axes[0, i]
        sub = df_valid[df_valid["model"] == mname].copy()
        sub["rankme_rank"] = sub["rankme"].rank()
        sub["mmlu_rank"]   = sub["mmlu_accuracy"].rank()

        ax.scatter(sub["rankme_rank"], sub["mmlu_rank"],
                   s=80, alpha=0.85, color=color_cycle[i])

        for _, row in sub.iterrows():
            ax.annotate(row["language"],
                        (row["rankme_rank"], row["mmlu_rank"]),
                        xytext=(4, 3), textcoords="offset points", fontsize=8)

        max_rank = max(sub["rankme_rank"].max(), sub["mmlu_rank"].max())
        ax.plot([1, max_rank], [1, max_rank], "k--", alpha=0.3, linewidth=1,
                label="perfect agreement")

        sr = corr_results.get(mname, {}).get("spearman_r", float("nan"))
        ax.set_title(f"{mname}\nSpearman r = {sr:+.3f}", fontsize=10)
        ax.set_xlabel("RankMe rank  (1 = lowest RankMe)", fontsize=9)
        ax.set_ylabel("M-MMLU rank  (1 = lowest accuracy)", fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.suptitle(
        f"Language rank by RankMe vs M-MMLU — {rankme_cfg['layer']}, "
        f"{rankme_cfg['aggregation']}",
        fontsize=12, y=1.02
    )
    plt.tight_layout()
    fig_path = figures_dir / (
        f"rankme_vs_mmmlu_rank"
        f"_{rankme_cfg['layer']}_{rankme_cfg['aggregation']}.pdf"
    )
    plt.savefig(fig_path, bbox_inches="tight")
    plt.show()
    print(f"Saved → {fig_path}")

## 5. Layer-wise Correlation Analysis

How does the Spearman correlation between RankMe and M-MMLU accuracy vary across
**layer depth** and **aggregation method**?

This helps identify which layers' representations are most predictive of downstream
multilingual performance — and whether mean-pooled or last-token representations work better.

In [ ]:
# ── Build layer × aggregation × language correlation table ─────────────────────
# Only makes sense if we have MMLU results for at least some languages
models_with_mmlu = [
    mcfg["name"] for mcfg in models_cfg
    if any(
        (mmlu_results.get(mcfg["name"]) or {}).get(l["name"], {}).get("accuracy") is not None
        for l in languages
    )
]

layer_corr_rows = []
for mcfg in models_cfg:
    mname = mcfg["name"]
    if mname not in models_with_mmlu:
        print(f"[{mname}] No MMLU results — skipping layer analysis.")
        continue

    df_all = load_all_rankme(mcfg, rankme_cfg["checkpoint"], PROJECT_ROOT)
    if df_all.empty:
        print(f"[{mname}] No all-layer RankMe data.")
        continue

    mmlu_map = {
        l["name"]: (mmlu_results.get(mname, {}).get(l["name"]) or {}).get("accuracy")
        for l in languages
    }

    for (layer, agg), grp in df_all.groupby(["layer", "aggregation"]):
        pairs = [
            (row["rankme"], mmlu_map.get(row["dataset"]))
            for _, row in grp.iterrows()
            if mmlu_map.get(row["dataset"]) is not None
        ]
        if len(pairs) < 3:
            continue
        rm_vals, mm_vals = zip(*pairs)
        sr, sp = stats.spearmanr(rm_vals, mm_vals)
        pr, pp = stats.pearsonr(rm_vals, mm_vals)

        # Extract numeric layer index for sorting
        layer_idx = int(layer.replace("layer_", "")) if "layer_" in str(layer) else -1

        layer_corr_rows.append({
            "model":      mname,
            "layer":      layer,
            "layer_idx":  layer_idx,
            "aggregation": agg,
            "spearman_r": sr,
            "pearson_r":  pr,
            "n":          len(pairs),
        })

df_layer = pd.DataFrame(layer_corr_rows)

if df_layer.empty:
    print("No layer-wise data available yet (run M-MMLU evaluation first).")
else:
    print(f"Layer correlation table: {len(df_layer)} rows")
    print(df_layer.sort_values(["model", "aggregation", "layer_idx"]).to_string(index=False))

In [ ]:
# ── Plot: Spearman r vs layer depth, per model × aggregation ──────────────────
if not df_layer.empty:
    models_in_layer = df_layer["model"].unique()
    n_models        = len(models_in_layer)
    agg_styles      = {"mean": "-o", "last": "--s", "first": ":^"}

    fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 4), squeeze=False)

    for i, mname in enumerate(models_in_layer):
        ax   = axes[0, i]
        sub  = df_layer[df_layer["model"] == mname].sort_values("layer_idx")

        for agg, grp in sub.groupby("aggregation"):
            grp_sorted = grp.sort_values("layer_idx")
            style = agg_styles.get(agg, "-o")
            ax.plot(grp_sorted["layer_idx"], grp_sorted["spearman_r"],
                    style, label=agg, markersize=6)

        ax.axhline(0, color="gray", linewidth=0.8, linestyle=":")

        # Mark configured layer
        cfg_idx = int(rankme_cfg["layer"].replace("layer_", ""))
        ax.axvline(cfg_idx, color="red", linewidth=1, linestyle="--",
                   label=f"config: {rankme_cfg['layer']}")

        ax.set_title(mname, fontsize=11)
        ax.set_xlabel("Layer index", fontsize=9)
        ax.set_ylabel("Spearman r (RankMe ↔ M-MMLU)", fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.suptitle(
        f"Spearman correlation: RankMe ↔ M-MMLU by layer"
        f"  (ckpt: {rankme_cfg['checkpoint']})",
        fontsize=12, y=1.02
    )
    plt.tight_layout()
    fig_path = figures_dir / "rankme_vs_mmmlu_by_layer.pdf"
    plt.savefig(fig_path, bbox_inches="tight")
    plt.show()
    print(f"Saved → {fig_path}")

In [ ]:
# ── Heatmap: Spearman r by layer × aggregation (one heatmap per model) ─────────
if not df_layer.empty:
    for mname in df_layer["model"].unique():
        sub = df_layer[df_layer["model"] == mname]
        pivot = sub.pivot_table(
            values="spearman_r", index="layer_idx", columns="aggregation"
        ).sort_index()

        fig, ax = plt.subplots(figsize=(4, max(4, len(pivot) * 0.4)))
        sns.heatmap(
            pivot, ax=ax, annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            linewidths=0.5, cbar_kws={"label": "Spearman r"}
        )
        ax.set_title(f"{mname} — Spearman r (RankMe ↔ M-MMLU)", fontsize=11)
        ax.set_xlabel("Aggregation")
        ax.set_ylabel("Layer index")
        plt.tight_layout()
        fig_path = figures_dir / (
            f"layer_heatmap_{mname.lower().replace(' ', '_').replace('/', '_')}.pdf"
        )
        plt.savefig(fig_path, bbox_inches="tight")
        plt.show()
        print(f"Saved → {fig_path}")

## 6. Save Results

In [ ]:
# ── Correlation summary JSON ───────────────────────────────────────────────────
summary_path = results_dir / "correlation_summary.json"
summary = {
    "rankme_config": {
        "checkpoint":  rankme_cfg["checkpoint"],
        "layer":       rankme_cfg["layer"],
        "aggregation": rankme_cfg["aggregation"],
    },
    "correlations": corr_results,
}
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"Correlation summary → {summary_path}")

# ── Layer correlation CSV ──────────────────────────────────────────────────────
if not df_layer.empty:
    layer_path = results_dir / "layer_correlation.csv"
    df_layer.drop(columns=["layer_idx"]).to_csv(layer_path, index=False)
    print(f"Layer correlation  → {layer_path}")

# ── Final summary table ────────────────────────────────────────────────────────
print("\n" + "="*55)
print("SUMMARY")
print("="*55)
print(f"  Checkpoint : {rankme_cfg['checkpoint']}")
print(f"  Layer      : {rankme_cfg['layer']}")
print(f"  Aggregation: {rankme_cfg['aggregation']}")
print()
print(f"{'Model':<20}  {'N':>3}  {'Pearson r':>10}  {'Spearman r':>10}")
print("-" * 50)
for mname, cr in corr_results.items():
    if mname == "Pooled":
        continue
    print(f"{mname:<20}  {cr['n']:>3}  "
          f"{cr['pearson_r']:>+10.4f}  {cr['spearman_r']:>+10.4f}")
if "Pooled" in corr_results:
    cr = corr_results["Pooled"]
    print("-" * 50)
    print(f"{'Pooled':<20}  {cr['n']:>3}  "
          f"{cr['pearson_r']:>+10.4f}  {cr['spearman_r']:>+10.4f}")